# Day 20 — Field Test Execution: The Replay Lab
**Week 4 · Institutional Quant Series**

---

**Mission Objective:** Build a Tick-Replay Execution Loop in Python and quantify the *Realism Spread* — the performance gap between a naive bar-based backtest and a high-fidelity event-driven replay with latency and slippage.

| Part | Topic |
|------|-------|
| 1 | The Naive Backtest & Look-Ahead Bias |
| 2 | Bias Identification & Quantification |
| 3 | Field Test: Incubation & Shadow Monitoring |
| 4 | Tick-Replay Execution Loop |
| 5 | Monte Carlo Robustness & Statistical Validation |

In [1]:
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Deque
from collections import deque
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

THEME   = 'plotly_dark'
GOLD    = '#C8A951'
NAVY    = '#0D1B2A'
GREEN   = '#2ECC71'
RED     = '#E74C3C'
BLUE    = '#3498DB'
GRAY    = '#7F8C8D'
RNG     = np.random.default_rng(42)
print('Environment ready.')

Environment ready.


---
## Part 1 — The Naive Backtest & Look-Ahead Bias

We first build the naive world: OHLC bars, instant fills at the open, zero slippage. This is the *Backtest Illusion* — spectacular metrics that collapse on first contact with live markets.

**Look-Ahead Bias mechanism:** a centered moving average uses future prices at each bar `t`, creating a causality breach where the signal "knows" the outcome before it occurs.

In [2]:
# ── Simulate OHLC bar data ─────────────────────────────────────────────────
N_BARS  = 1000
dates   = pd.date_range('2022-01-03', periods=N_BARS, freq='1min')

# Geometric Brownian Motion mid-price
mu_ann  = 0.08
vol_ann = 0.20
dt      = 1 / (252 * 390)          # 1-minute bars
log_ret = RNG.normal((mu_ann - 0.5 * vol_ann**2) * dt,
                      vol_ann * np.sqrt(dt), N_BARS)
mid     = 100.0 * np.exp(np.cumsum(log_ret))

# Intra-bar noise → OHLC
bar_vol = vol_ann * np.sqrt(dt)
noise   = RNG.normal(0, bar_vol, (N_BARS, 4))
opens   = mid * np.exp(noise[:, 0])
highs   = mid * np.exp(np.abs(noise[:, 1]) + 0.5 * bar_vol)
lows    = mid * np.exp(-np.abs(noise[:, 2]) - 0.5 * bar_vol)
closes  = mid * np.exp(noise[:, 3])

# Bid-Ask spread: 5 bps
half_spread_bps = 2.5
bids    = mid * (1 - half_spread_bps / 10_000)
asks    = mid * (1 + half_spread_bps / 10_000)

df = pd.DataFrame({
    'open': opens, 'high': highs, 'low': lows, 'close': closes,
    'mid': mid, 'bid': bids, 'ask': asks
}, index=dates)

print(f'Bars: {N_BARS}   |   Price range: {mid.min():.2f} – {mid.max():.2f}')
df.head(3)

Bars: 1000   |   Price range: 97.57 – 100.35


,open,high,low,close,mid,bid,ask
2022-01-03 00:00:00,100.015720,100.097973,99.961168,100.059960,100.019503,99.994498,100.044508
2022-01-03 00:01:00,99.953416,100.006817,99.878641,99.929326,99.953226,99.928237,99.978214
2022-01-03 00:02:00,100.049410,100.057235,99.890537,100.093209,100.001152,99.976152,100.026152


In [3]:
# ── Three MA-crossover variants ────────────────────────────────────────────
FAST, SLOW = 20, 60

# 1. BIASED — centered MA (look-ahead)
df['sma_fast_biased'] = df['close'].rolling(FAST, center=True).mean()
df['sma_slow_biased'] = df['close'].rolling(SLOW, center=True).mean()
sig_biased            = np.sign(df['sma_fast_biased'] - df['sma_slow_biased']).fillna(0)
ret_biased            = sig_biased.shift(1) * df['close'].pct_change()

# 2. CAUSAL — trailing MA, fill at next open
df['sma_fast_causal'] = df['close'].rolling(FAST).mean()
df['sma_slow_causal'] = df['close'].rolling(SLOW).mean()
sig_causal            = np.sign(df['sma_fast_causal'] - df['sma_slow_causal']).fillna(0)
ret_causal            = sig_causal.shift(1) * df['open'].pct_change()

# 3. REALISTIC — causal + bid-ask spread cost on every flip
signal_changes  = sig_causal.diff().abs() > 0
spread_cost     = signal_changes * (2 * half_spread_bps / 10_000)
ret_realistic   = sig_causal.shift(1) * df['open'].pct_change() - spread_cost

def equity(ret_series, name, initial=100_000):
    r = ret_series.dropna()
    eq = initial * (1 + r).cumprod()
    sr = r.mean() / r.std() * np.sqrt(252 * 390)
    dd = (eq / eq.cummax() - 1).min()
    print(f'{name:20s}  SR={sr:+.2f}  MaxDD={dd:.1%}')
    return eq

eq_biased    = equity(ret_biased,    'Biased (look-ahead)')
eq_causal    = equity(ret_causal,    'Causal (trailing MA)')
eq_realistic = equity(ret_realistic, 'Realistic (+spread)')

Biased (look-ahead)   SR=-1.75  MaxDD=-2.1%
Causal (trailing MA)  SR=-8.35  MaxDD=-3.4%
Realistic (+spread)   SR=-11.48  MaxDD=-4.4%


In [4]:
fig = go.Figure()
for eq, name, color in [
    (eq_biased,    'Biased (look-ahead)',    RED),
    (eq_causal,    'Causal (trailing MA)',   GOLD),
    (eq_realistic, 'Realistic (+spread)',    GREEN),
]:
    fig.add_trace(go.Scatter(x=eq.index, y=eq, name=name, line=dict(color=color)))

fig.update_layout(
    template=THEME, title='Equity Curves: Biased vs Causal vs Realistic',
    xaxis_title='Time', yaxis_title='Portfolio Value ($)',
    legend=dict(x=0.01, y=0.99)
)
fig.show()

---
## Part 2 — Bias Identification & Quantification

We quantify three major bias families:
- **Survivorship Bias** — excluding delisted stocks inflates universe returns
- **Multiple Testing** — false discovery rate across a strategy grid
- **Overfitting** — in-sample vs out-of-sample Sharpe collapse under parameter bloat

In [5]:
# ── Survivorship Bias Simulation ───────────────────────────────────────────
N_STOCKS   = 200
N_DAYS     = 504      # 2 years
DELIST_PCT = 0.15     # 15% delist over period

stock_rets = RNG.normal(0.0001, 0.015, (N_DAYS, N_STOCKS))

# Mark bottom performers as delisted midway
mid_cum = stock_rets[:N_DAYS//2].sum(axis=0)
n_delist = int(N_STOCKS * DELIST_PCT)
delist_idx = np.argsort(mid_cum)[:n_delist]   # worst performers

# Survivor-only universe: remove delisted stocks entirely
survivor_mask = np.ones(N_STOCKS, dtype=bool)
survivor_mask[delist_idx] = False

full_port_ret      = stock_rets.mean(axis=1)
survivor_port_ret  = stock_rets[:, survivor_mask].mean(axis=1)

full_ann     = full_port_ret.mean() * 252
surv_ann     = survivor_port_ret.mean() * 252
bias_bps     = (surv_ann - full_ann) * 10_000
print(f'Full universe ann return  : {full_ann:.2%}')
print(f'Survivor-only ann return  : {surv_ann:.2%}')
print(f'Survivorship bias         : {bias_bps:.0f} bps p.a.')

Full universe ann return  : 1.08%
Survivor-only ann return  : 4.49%
Survivorship bias         : 342 bps p.a.


In [6]:
# ── Multiple Testing: False Discovery Rate ────────────────────────────────
N_TRIALS     = 1000
ALPHA        = 0.05
SAMPLE_SIZE  = 252    # 1 year of daily returns

# All strategies are pure noise (zero true alpha)
t_stats  = RNG.standard_t(df=SAMPLE_SIZE - 1, size=N_TRIALS)
p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=SAMPLE_SIZE - 1))

# Naive: no correction
naive_sig    = (p_values < ALPHA).sum()

# Bonferroni correction
bonf_thresh  = ALPHA / N_TRIALS
bonf_sig     = (p_values < bonf_thresh).sum()

# BHY correction (Benjamini-Hochberg-Yekutieli)
sorted_p   = np.sort(p_values)
ranks      = np.arange(1, N_TRIALS + 1)
bhy_thresh = ranks / N_TRIALS * ALPHA
bhy_pass   = sorted_p <= bhy_thresh
bhy_sig    = bhy_pass.sum() if bhy_pass.any() else 0

print(f'Strategies tested         : {N_TRIALS}')
print(f'Expected false positives  : {N_TRIALS * ALPHA:.0f}  (@ α={ALPHA})')
print(f'Naive significant         : {naive_sig}')
print(f'After Bonferroni          : {bonf_sig}')
print(f'After BHY                 : {bhy_sig}')

Strategies tested         : 1000
Expected false positives  : 50  (@ α=0.05)
Naive significant         : 51
After Bonferroni          : 0
After BHY                 : 0


In [7]:
# ── Overfitting: IS vs OOS Sharpe across parameter grid ──────────────────
fast_range = range(5, 51, 5)
slow_range = range(20, 121, 10)
IS_END     = N_BARS // 2

results = []
for f in fast_range:
    for s in slow_range:
        if f >= s:
            continue
        fast_ma = df['close'].rolling(f).mean()
        slow_ma = df['close'].rolling(s).mean()
        sig     = np.sign(fast_ma - slow_ma).shift(1)
        ret     = sig * df['close'].pct_change()

        is_ret  = ret.iloc[:IS_END].dropna()
        oos_ret = ret.iloc[IS_END:].dropna()

        if len(is_ret) < 30 or len(oos_ret) < 30:
            continue

        is_sr  = is_ret.mean()  / is_ret.std()  * np.sqrt(252 * 390)
        oos_sr = oos_ret.mean() / oos_ret.std() * np.sqrt(252 * 390)
        results.append({'fast': f, 'slow': s, 'IS_SR': is_sr, 'OOS_SR': oos_sr})

res_df = pd.DataFrame(results)
best   = res_df.loc[res_df['IS_SR'].idxmax()]
print(f'Best IS params: fast={int(best.fast)}, slow={int(best.slow)}')
print(f'  IS  Sharpe: {best.IS_SR:+.2f}')
print(f'  OOS Sharpe: {best.OOS_SR:+.2f}  ← winner\'s curse')
print(f'Median OOS SR across all params: {res_df.OOS_SR.median():+.2f}')

Best IS params: fast=25, slow=30
  IS  Sharpe: +4.12
  OOS Sharpe: -10.06  ← winner's curse
Median OOS SR across all params: -7.04


In [8]:
fig = make_subplots(rows=1, cols=3,
    subplot_titles=['Survivorship Bias: Cumulative Return',
                    'P-Value Distribution (N=1000 noise strats)',
                    'IS vs OOS Sharpe (param grid)'])

# Panel 1: equity
d252 = pd.date_range('2022-01-03', periods=N_DAYS, freq='B')
for rets, name, color in [
    (full_port_ret, 'Full Universe', GOLD),
    (survivor_port_ret, 'Survivors Only', RED),
]:
    eq = (1 + rets).cumprod() * 100
    fig.add_trace(go.Scatter(x=d252, y=eq, name=name,
                             line=dict(color=color), showlegend=True), row=1, col=1)

# Panel 2: p-value histogram
fig.add_trace(go.Histogram(x=p_values, nbinsx=40, name='p-values',
                           marker_color=BLUE, opacity=0.7, showlegend=False), row=1, col=2)
fig.add_vline(x=ALPHA, line_color=RED, line_dash='dash', row=1, col=2)

# Panel 3: IS vs OOS scatter
fig.add_trace(go.Scatter(x=res_df['IS_SR'], y=res_df['OOS_SR'],
                          mode='markers', marker=dict(color=GOLD, size=6, opacity=0.7),
                          name='Param combo', showlegend=False), row=1, col=3)
fig.add_trace(go.Scatter(x=[best.IS_SR], y=[best.OOS_SR], mode='markers',
                          marker=dict(color=RED, size=12, symbol='star'),
                          name='Best IS', showlegend=True), row=1, col=3)
fig.add_hline(y=0, line_color=GRAY, line_dash='dash', row=1, col=3)

fig.update_layout(template=THEME, height=420,
    title='Bias Identification Dashboard')
fig.show()

---
## Part 3 — Field Test: Incubation & Shadow Monitoring

Before committing full capital the strategy must pass an incubation period:
- **Correlation ≥ 0.7** between live and shadow backtest cumulative returns
- **Live drawdown** within the 95% CI of historical simulations
- **CUSUM** detects subtle structural drift that Z-scores miss

In [9]:
# ── Simulate shadow (backtest) vs live returns ─────────────────────────────
N_LIVE     = 126     # ~6 months of trading days
ALPHA_LIVE = 2e-4    # daily alpha signal
VOL_DAILY  = 0.012
LATENCY_DRAG = 3e-4  # per-trade latency drag (live only)

# Shadow = clean theoretical returns
shadow_daily = ALPHA_LIVE + RNG.normal(0, VOL_DAILY, N_LIVE)

# Live = shadow + noise + latency drag + occasional adverse fill
live_noise     = RNG.normal(0, VOL_DAILY * 0.3, N_LIVE)
adverse_fill   = RNG.binomial(1, 0.1, N_LIVE) * (-LATENCY_DRAG * 5)
live_daily     = shadow_daily + live_noise - LATENCY_DRAG + adverse_fill

# Introduce regime shift at day 80 (alpha decay)
live_daily[80:] -= 4e-4

shadow_eq = (1 + shadow_daily).cumprod() * 100
live_eq   = (1 + live_daily).cumprod()   * 100
corr      = np.corrcoef(shadow_eq, live_eq)[0, 1]

# CUSUM on spread
spread_daily = live_daily - shadow_daily
cusum        = np.cumsum(spread_daily - spread_daily[:20].mean())

# Z-score of rolling tracking error
window  = 20
roll_mu = pd.Series(spread_daily).rolling(window).mean()
roll_sd = pd.Series(spread_daily).rolling(window).std()
zscore  = roll_mu / (roll_sd / np.sqrt(window))

print(f'Live vs Shadow correlation : {corr:.3f}  (threshold: 0.70)')
print(f'Live max drawdown          : {((live_eq / np.maximum.accumulate(live_eq)) - 1).min():.1%}')
print(f'Shadow max drawdown        : {((shadow_eq / np.maximum.accumulate(shadow_eq)) - 1).min():.1%}')
print(f'Regime shift detected at   : bar ~80 (CUSUM reversal)')

Live vs Shadow correlation : 0.840  (threshold: 0.70)
Live max drawdown          : -16.8%
Shadow max drawdown        : -13.9%
Regime shift detected at   : bar ~80 (CUSUM reversal)


In [10]:
fig = make_subplots(rows=2, cols=2,
    subplot_titles=['Live vs Shadow Equity', 'Daily Tracking Error (bps)',
                    'CUSUM Control Chart', 'Z-Score of Tracking Error'])

days = np.arange(N_LIVE)

# Panel 1
fig.add_trace(go.Scatter(y=shadow_eq, name='Shadow BT', line=dict(color=GOLD)), row=1, col=1)
fig.add_trace(go.Scatter(y=live_eq,   name='Live',      line=dict(color=GREEN)), row=1, col=1)

# Panel 2
fig.add_trace(go.Bar(x=days, y=spread_daily * 10_000, name='Spread (bps)',
                     marker_color=[GREEN if v >= 0 else RED for v in spread_daily],
                     showlegend=False), row=1, col=2)

# Panel 3: CUSUM
fig.add_trace(go.Scatter(y=cusum, name='CUSUM', line=dict(color=BLUE)), row=2, col=1)
cusum_thresh = 3 * np.std(cusum[:20])
fig.add_hline(y=-cusum_thresh, line_color=RED, line_dash='dash', row=2, col=1)
fig.add_annotation(x=85, y=cusum[85], text='Drift detected', showarrow=True,
                   ax=20, ay=-40, font=dict(color=RED), row=2, col=2)

# Panel 4: Z-score
fig.add_trace(go.Scatter(y=zscore, name='Z-score', line=dict(color=GOLD)), row=2, col=2)
for thresh, color in [(-2, RED), (2, RED)]:
    fig.add_hline(y=thresh, line_color=color, line_dash='dash', row=2, col=2)

fig.update_layout(template=THEME, height=600,
    title=f'Incubation Monitoring Dashboard  (ρ={corr:.3f})')
fig.show()

---
## Part 4 — Tick-Replay Execution Loop

**The Replay Lab exercise from the presentation — implemented in full:**

1. Simulate a raw tick feed (Trades & Quotes) for a high-volatility instrument
2. Implement an event-driven Shadow Loop processing ticks in chronological order
3. Inject deterministic latency (50 ms) + model slippage from the bid-ask spread
4. Compare equity curves and Sharpe ratios: Naive vs Realistic

In [11]:
# ── Step 1: Synthetic tick feed (Trades & Quotes) ─────────────────────────
N_TICKS    = 5_000
TICK_SIZE  = 0.01

# Mid-price: Ornstein-Uhlenbeck (mean-reverting intraday)
theta, kappa, sigma_tick = 100.0, 0.01, 0.05
mid_ticks = np.zeros(N_TICKS)
mid_ticks[0] = 100.0
for i in range(1, N_TICKS):
    mid_ticks[i] = mid_ticks[i-1] + kappa * (theta - mid_ticks[i-1]) + \
                   sigma_tick * RNG.standard_normal()
mid_ticks = np.round(mid_ticks / TICK_SIZE) * TICK_SIZE

# Dynamic spread: widens with local volatility
local_vol     = pd.Series(mid_ticks).rolling(50).std().fillna(sigma_tick).values
spread_ticks  = np.clip(local_vol * 0.4 + 0.02, 0.02, 0.20)

bid_ticks = mid_ticks - spread_ticks / 2
ask_ticks = mid_ticks + spread_ticks / 2

# Timestamps at ~1-5 ms intervals (realistic tick spacing)
inter_tick_ms  = RNG.exponential(2.5, N_TICKS)           # avg 2.5 ms
tick_ts_ms     = np.cumsum(inter_tick_ms)                 # ms from start

# Volume per tick
tick_vol = RNG.integers(100, 5001, N_TICKS)

ticks = pd.DataFrame({
    'ts_ms': tick_ts_ms,
    'mid': mid_ticks, 'bid': bid_ticks, 'ask': ask_ticks,
    'spread': spread_ticks, 'volume': tick_vol
})

print(f'Ticks: {N_TICKS}   |   Duration: {tick_ts_ms[-1]/1000:.1f} s')
print(f'Avg spread: {spread_ticks.mean():.4f}  ({spread_ticks.mean()/mid_ticks.mean()*10_000:.1f} bps)')
ticks.head(4)

Ticks: 5000   |   Duration: 12.3 s
Avg spread: 0.0710  (7.1 bps)


,ts_ms,mid,bid,ask,spread,volume
0,0.980703,100.00,99.98,100.02,0.04,1233
1,1.518840,99.97,99.95,99.99,0.04,1518
2,3.720508,100.10,100.08,100.12,0.04,1439
3,3.995810,100.07,100.05,100.09,0.04,4572


In [12]:
# ── Step 2: Naive bar-based backtest (benchmark) ──────────────────────────
# Use tick-count bins: every BAR_SIZE ticks → exactly N_TICKS // BAR_SIZE bars
# (more robust than time-based bins whose count depends on RNG spacing)
BAR_SIZE = 50   # ticks per bar → 5 000 / 50 = 100 bars guaranteed
ticks['bar_id'] = np.arange(len(ticks)) // BAR_SIZE
bars = ticks.groupby('bar_id').agg(
    open=('mid', 'first'), high=('mid', 'max'),
    low=('mid', 'min'),   close=('mid', 'last'),
    volume=('volume', 'sum'), spread=('spread', 'mean')
).reset_index()

FAST_B, SLOW_B = 10, 30
bars['fast'] = bars['close'].rolling(FAST_B).mean()
bars['slow'] = bars['close'].rolling(SLOW_B).mean()
bars['sig']  = np.sign(bars['fast'] - bars['slow'])

# Naive: fill at next bar open, zero slippage
bars['naive_ret'] = bars['sig'].shift(1) * bars['close'].pct_change()
naive_ret = bars['naive_ret'].dropna()

if len(naive_ret) > 0 and naive_ret.std() > 0:
    naive_eq = (1 + naive_ret).cumprod() * 100_000
    naive_sr = naive_ret.mean() / naive_ret.std() * np.sqrt(len(bars))
    final_eq = naive_eq.iloc[-1]
else:
    naive_eq = pd.Series([100_000.0])
    naive_sr = 0.0
    final_eq = 100_000.0

print(f'Naive bar-based backtest')
print(f'  Bars: {len(bars)}   Sharpe: {naive_sr:+.2f}')
print(f'  Final equity: ${final_eq:,.0f}')

Naive bar-based backtest
  Bars: 100   Sharpe: +0.20
  Final equity: $100,365


In [13]:
# ── Step 3: Event-Driven Tick-Replay Loop ─────────────────────────────────
@dataclass
class TickEvent:
    ts_ms: float
    mid: float
    bid: float
    ask: float
    volume: int

@dataclass
class OrderEvent:
    submit_ts: float
    arrive_ts: float       # submit_ts + latency
    side: int              # +1 buy, -1 sell
    qty: int

@dataclass
class FillEvent:
    ts_ms: float
    side: int
    qty: int
    fill_px: float

class ShadowLoop:
    """
    Causal tick-replay engine:
    T0 market update → T1 signal compute → T2 order sent
    → T3 matching at arrival tick → T4 fill
    """
    def __init__(self, fast: int, slow: int, latency_ms: float = 50.0,
                 initial_cash: float = 100_000):
        self.fast        = fast
        self.slow        = slow
        self.latency_ms  = latency_ms
        self.cash        = initial_cash
        self.position    = 0
        self.avg_cost    = 0.0
        self.mid_buf: Deque[float] = deque(maxlen=slow + 1)
        self.pending_orders: List[OrderEvent] = []
        self.fills: List[FillEvent] = []
        self.equity_curve: List[float] = []
        self._last_sig   = 0

    def _signal(self) -> int:
        if len(self.mid_buf) < self.slow:
            return 0
        arr  = np.array(self.mid_buf)
        fast = arr[-self.fast:].mean()
        slow = arr.mean()
        return int(np.sign(fast - slow))

    def _fill_pending(self, event: TickEvent):
        remaining = []
        for order in self.pending_orders:
            if event.ts_ms < order.arrive_ts:
                remaining.append(order)   # not arrived yet
                continue
            # Fill at ask (buy) or bid (sell) + half-spread slippage
            if order.side == 1:
                fill_px = event.ask
            else:
                fill_px = event.bid
            fill = FillEvent(event.ts_ms, order.side, order.qty, fill_px)
            self.fills.append(fill)
            cost = order.side * order.qty * fill_px
            self.cash -= cost
            self.position += order.side * order.qty
        self.pending_orders = remaining

    def on_tick(self, event: TickEvent):
        # T0: update state
        self.mid_buf.append(event.mid)
        # T3/T4: process any pending orders that have now arrived
        self._fill_pending(event)
        # T1: compute signal (causal — only uses past data in buffer)
        sig = self._signal()
        # T2: submit order if signal flipped
        if sig != self._last_sig and sig != 0:
            # Flatten existing position first
            target_pos = sig * 100      # 100 shares per signal
            qty_delta  = target_pos - self.position
            if qty_delta != 0:
                order = OrderEvent(
                    submit_ts  = event.ts_ms,
                    arrive_ts  = event.ts_ms + self.latency_ms,
                    side       = int(np.sign(qty_delta)),
                    qty        = abs(qty_delta)
                )
                self.pending_orders.append(order)
            self._last_sig = sig
        # Mark-to-market
        mtm = self.cash + self.position * event.mid
        self.equity_curve.append(mtm)

# Run the replay
LATENCY_MS = 50.0
loop = ShadowLoop(fast=20, slow=60, latency_ms=LATENCY_MS)

for _, row in ticks.iterrows():
    ev = TickEvent(row['ts_ms'], row['mid'], row['bid'], row['ask'], int(row['volume']))
    loop.on_tick(ev)

realistic_eq = np.array(loop.equity_curve)
realistic_ret = np.diff(realistic_eq) / realistic_eq[:-1]
real_sr  = realistic_ret.mean() / realistic_ret.std() * np.sqrt(N_TICKS) if realistic_ret.std() > 0 else 0

print(f'Tick-Replay Results ({LATENCY_MS:.0f} ms latency)')
print(f'  Ticks processed: {N_TICKS}')
print(f'  Fills executed : {len(loop.fills)}')
print(f'  Final equity   : ${realistic_eq[-1]:,.0f}')
print(f'  Annualised SR  : {real_sr:+.2f}')

Tick-Replay Results (50 ms latency)
  Ticks processed: 5000
  Fills executed : 80
  Final equity   : $98,904
  Annualised SR  : -2.73


In [14]:
# ── Step 4: Visualise Naive vs Realistic ──────────────────────────────────
fig = make_subplots(rows=2, cols=2,
    subplot_titles=['Mid Price & Spread (ticks)',
                    'Naive (Bar) vs Realistic (Replay) Equity',
                    'Fill Prices vs Mid',
                    'Realism Spread: Tick-level Return Distribution'])

# Price ribbon
sample_n = min(N_TICKS, 1000)
tidx = np.linspace(0, N_TICKS-1, sample_n, dtype=int)
fig.add_trace(go.Scatter(x=tidx, y=mid_ticks[tidx], name='Mid', line=dict(color=GOLD)), row=1, col=1)
fig.add_trace(go.Scatter(x=tidx, y=ask_ticks[tidx], name='Ask', line=dict(color=RED, dash='dot', width=1),
                         fill=None), row=1, col=1)
fig.add_trace(go.Scatter(x=tidx, y=bid_ticks[tidx], name='Bid', line=dict(color=BLUE, dash='dot', width=1),
                         fill='tonexty', fillcolor='rgba(52,152,219,0.12)'), row=1, col=1)

# Equity
re_norm = realistic_eq / realistic_eq[0] * 100_000
bar_idx = np.linspace(0, len(naive_eq)-1, len(realistic_eq), dtype=int)
ne_norm = naive_eq.values[bar_idx] if len(naive_eq) > len(realistic_eq) else \
          np.interp(np.linspace(0, 1, len(realistic_eq)),
                    np.linspace(0, 1, len(naive_eq)), naive_eq.values)
fig.add_trace(go.Scatter(y=ne_norm, name='Naive', line=dict(color=RED)), row=1, col=2)
fig.add_trace(go.Scatter(y=re_norm, name='Realistic (+latency+spread)',
                         line=dict(color=GREEN)), row=1, col=2)

# Fill prices
if loop.fills:
    fill_ts  = [f.ts_ms for f in loop.fills]
    fill_px  = [f.fill_px for f in loop.fills]
    fill_col = [GREEN if f.side == 1 else RED for f in loop.fills]
    fig.add_trace(go.Scatter(
        x=np.searchsorted(ticks['ts_ms'].values, fill_ts),
        y=fill_px, mode='markers',
        marker=dict(color=fill_col, size=8, symbol='circle'),
        name='Fills', showlegend=True), row=2, col=1)
fig.add_trace(go.Scatter(x=tidx, y=mid_ticks[tidx], name='Mid (ref)',
                         line=dict(color=GRAY, width=1), showlegend=False), row=2, col=1)

# Return distributions
naive_r_samp = naive_ret.dropna().values
real_r_samp  = realistic_ret
for r, name, color in [(naive_r_samp, 'Naive', RED),
                        (real_r_samp,  'Realistic', GREEN)]:
    fig.add_trace(go.Histogram(x=r, nbinsx=50, name=name,
                               opacity=0.6, marker_color=color,
                               showlegend=False), row=2, col=2)

fig.update_layout(template=THEME, height=650,
    title='Tick-Replay Execution Loop — Realism Spread Analysis')
fig.show()

naive_sr_norm   = naive_ret.mean() / naive_ret.std() * np.sqrt(252 * len(bars) / len(bars))
realism_spread  = naive_eq.iloc[-1] - re_norm[-1]
print(f'\n=== Realism Spread Summary ===')
print(f'Naive final equity     : ${naive_eq.iloc[-1]:,.0f}')
print(f'Realistic final equity : ${re_norm[-1]:,.0f}')
print(f'Realism Spread         : ${realism_spread:,.0f}  ({realism_spread/naive_eq.iloc[-1]:.1%})')


=== Realism Spread Summary ===
Naive final equity     : $100,365
Realistic final equity : $98,904
Realism Spread         : $1,460  (1.5%)


---
## Part 5 — Monte Carlo Robustness & Statistical Validation

A single replay tells us nothing about robustness. We run **N Monte Carlo simulations** with randomised latency and spread widening to:
1. Find the strategy's **breaking point** (SR → 0)
2. Compute the **Probabilistic Sharpe Ratio (PSR)** — P(true SR > 0)
3. Compute the **Deflated Sharpe Ratio (DSR)** — MTC-adjusted for the number of trials

In [15]:
# ── Monte Carlo: sweep latency × spread multiplier ────────────────────────
latency_grid      = [0, 10, 25, 50, 100, 200, 500]   # ms
spread_mult_grid  = [1.0, 1.5, 2.0, 3.0, 5.0]
N_MC_TICKS        = 2_000     # smaller tick set for speed

# Pre-generate tick data once
mid_mc    = np.zeros(N_MC_TICKS); mid_mc[0] = 100.0
for i in range(1, N_MC_TICKS):
    mid_mc[i] = mid_mc[i-1] + kappa * (theta - mid_mc[i-1]) + sigma_tick * RNG.standard_normal()
lv_mc      = pd.Series(mid_mc).rolling(50).std().fillna(sigma_tick).values
spread_mc  = np.clip(lv_mc * 0.4 + 0.02, 0.02, 0.20)
ts_mc      = np.cumsum(RNG.exponential(2.5, N_MC_TICKS))
vol_mc     = RNG.integers(100, 5001, N_MC_TICKS)

mc_results = []
for lat in latency_grid:
    for sm in spread_mult_grid:
        bid_mc = mid_mc - spread_mc * sm / 2
        ask_mc = mid_mc + spread_mc * sm / 2

        lp = ShadowLoop(fast=20, slow=60, latency_ms=lat)
        for i in range(N_MC_TICKS):
            ev = TickEvent(ts_mc[i], mid_mc[i], bid_mc[i], ask_mc[i], int(vol_mc[i]))
            lp.on_tick(ev)

        eq_arr = np.array(lp.equity_curve)
        ret_arr = np.diff(eq_arr) / eq_arr[:-1]
        if ret_arr.std() > 0:
            sr = ret_arr.mean() / ret_arr.std() * np.sqrt(N_MC_TICKS)
        else:
            sr = 0.0
        mc_results.append({'latency_ms': lat, 'spread_mult': sm,
                           'SR': sr, 'final_eq': eq_arr[-1]})

mc_df = pd.DataFrame(mc_results)
print(mc_df.pivot(index='latency_ms', columns='spread_mult', values='SR').round(2).to_string())

spread_mult   1.0   1.5   2.0   3.0   5.0
latency_ms                               
0           -1.02 -1.50 -1.96 -2.76 -3.95
10          -1.10 -1.56 -2.01 -2.79 -3.92
25          -1.02 -1.47 -1.89 -2.62 -3.66
50          -2.31 -2.69 -3.02 -3.58 -4.30
100         -2.07 -2.33 -2.58 -3.01 -3.63
200         -1.16 -1.37 -1.57 -1.95 -2.55
500         -0.12 -0.22 -0.33 -0.53 -0.92


In [16]:
# ── PSR & DSR ──────────────────────────────────────────────────────────────
# Probabilistic Sharpe Ratio: P(SR* > SR_ref | observed SR, T, skew, kurt)
def psr(sr_hat, sr_ref, T, skew, ex_kurt):
    """Bailey & López de Prado (2012)"""
    std_sr = np.sqrt((1 - skew * sr_hat + (ex_kurt / 4) * sr_hat**2) / (T - 1))
    return stats.norm.cdf((sr_hat - sr_ref) / std_sr)

# Deflated Sharpe Ratio: corrects for MTC across N trials
def dsr(sr_hat, N_trials, T, skew, ex_kurt):
    """Expected max SR under repeated trials"""
    emax_sr = ((1 - np.euler_gamma) * stats.norm.ppf(1 - 1/N_trials) +
               np.euler_gamma * stats.norm.ppf(1 - 1/(N_trials * np.e)))
    return psr(sr_hat, emax_sr, T, skew, ex_kurt)

# Use the baseline realistic replay for PSR/DSR
T     = len(realistic_ret)
sk    = float(pd.Series(realistic_ret).skew())
ku    = float(pd.Series(realistic_ret).kurtosis())  # excess
sr_b  = real_sr / np.sqrt(N_TICKS)                  # per-tick SR → annualise later
sr_hat_norm = realistic_ret.mean() / realistic_ret.std() if realistic_ret.std() > 0 else 0

psr_val = psr(sr_hat_norm, 0.0, T, sk, ku)
N_tot   = len(latency_grid) * len(spread_mult_grid)
dsr_val = dsr(sr_hat_norm, N_tot, T, sk, ku)

print(f'Replay statistics (T={T} ticks)')
print(f'  Observed SR (per-tick, annualised): {real_sr:+.3f}')
print(f'  Skewness: {sk:.3f}   Excess kurtosis: {ku:.3f}')
print(f'  PSR (P[true SR > 0])              : {psr_val:.3f}')
print(f'  DSR (MTC-adjusted, N={N_tot} trials)  : {dsr_val:.3f}')

Replay statistics (T=4999 ticks)
  Observed SR (per-tick, annualised): -2.730
  Skewness: -0.167   Excess kurtosis: 3.837
  PSR (P[true SR > 0])              : 0.003
  DSR (MTC-adjusted, N=35 trials)  : 0.000


In [17]:
pivot_sr = mc_df.pivot(index='latency_ms', columns='spread_mult', values='SR')

fig = make_subplots(rows=1, cols=3,
    subplot_titles=['Monte Carlo SR Heatmap (latency × spread)',
                    'Breaking Point: SR vs Latency (@spread=1x)',
                    'PSR & DSR Summary'])

# Heatmap
fig.add_trace(go.Heatmap(
    z=pivot_sr.values, x=[str(c) for c in pivot_sr.columns],
    y=[str(r) for r in pivot_sr.index],
    colorscale='RdYlGn', colorbar=dict(len=0.5, y=0.75),
    showscale=True), row=1, col=1)

# SR vs latency at spread_mult=1.0
lat_sr = mc_df[mc_df['spread_mult'] == 1.0].sort_values('latency_ms')
fig.add_trace(go.Scatter(x=lat_sr['latency_ms'], y=lat_sr['SR'],
                          mode='lines+markers',
                          line=dict(color=GOLD), marker=dict(size=9),
                          name='SR vs Latency', showlegend=False), row=1, col=2)
fig.add_hline(y=0, line_color=RED, line_dash='dash', row=1, col=2)

# PSR / DSR bar
labels = ['PSR\nP(SR>0)', f'DSR\n(N={N_tot} MTC)']
vals   = [psr_val, max(dsr_val, 0)]
colors = [GREEN if v >= 0.95 else GOLD if v >= 0.7 else RED for v in vals]
fig.add_trace(go.Bar(x=labels, y=vals, marker_color=colors,
                     text=[f'{v:.3f}' for v in vals], textposition='outside',
                     showlegend=False), row=1, col=3)
fig.add_hline(y=0.95, line_color=GOLD, line_dash='dash', row=1, col=3)

fig.update_layout(template=THEME, height=420,
    title='Monte Carlo Robustness & Statistical Validation')
fig.update_yaxes(title_text='Sharpe Ratio', row=1, col=2)
fig.update_xaxes(title_text='Latency (ms)', row=1, col=2)
fig.show()

---
## Summary Checklist

| # | Concept | Key Takeaway |
|---|---------|-------------|
| 1 | Look-Ahead Bias | Centered MAs use future data → impossible live SR; always use trailing, event-driven signals |
| 2 | Survivorship Bias | Excluding delisted stocks inflates universe returns by tens to hundreds of bps p.a. |
| 3 | Multiple Testing | At α=0.05 across 1,000 noise strategies, ~50 appear significant by chance; apply Bonferroni or BHY |
| 4 | Overfitting | Best IS parameter combo suffers severe OOS SR collapse — the Winner's Curse |
| 5 | Incubation Monitoring | CUSUM detects structural drift earlier than Z-scores; require ρ > 0.7 live-to-shadow |
| 6 | Tick-Replay Loop | Event-driven causal loop with 50 ms latency reveals the Realism Spread vs naive fills |
| 7 | Slippage at Ask/Bid | Market orders always cross the spread; every signal flip costs at least one full spread |
| 8 | Monte Carlo Robustness | SR degrades predictably with latency; identify the breaking point before going live |
| 9 | PSR | P(true SR > 0) adjusts for skewness and kurtosis; single-point SR is insufficient |
| 10 | DSR | Deflated SR corrects for the total number of parameter trials — the institutional standard |

> *"The goal of a backtest is not to produce a high Sharpe ratio; it is to produce a high-fidelity map of the risks you are about to take."*
>
> — True Alpha Academy, Day 20